In [ ]:
library(CellChat)
library(patchwork)
library(Seurat)
library(ggplot2)
library(RColorBrewer)

In [ ]:
data <- readRDS("../phaseZ_finalize_figs/250711_niches.rds")

In [ ]:
head(data@meta.data)

In [ ]:
sample_label_map <- c(
  "BS21-N65682A2__20241025" = "cellchat_case1",
  "BS22-T41795A1__20241025" = "cellchat_case2",
  "BS23_49001A1__20240803"  = "cellchat_case3",
  "BS23_52206A2__20240803"  = "cellchat_case4",
  
  "BS2_61615A1__20240803"   = "cellchat_control1",
  "BS22_12012A1__20240803"  = "cellchat_control2",
  "BS24-M35359A1__20241025" = "cellchat_control3",
  "BS24-R31519A2__20241025" = "cellchat_control4"
)

In [5]:
#Part I: Data input & processing and initialization of CellChat object
target_labels <- c("cellchat_control1", "cellchat_control2", "cellchat_control3", "cellchat_control4")
target_sample_ids <- names(sample_label_map)[sample_label_map %in% target_labels]
data_sub <- subset(data, subset = sample_id %in% target_sample_ids)

In [6]:
Idents(data_sub) <- "label_niches_fine"
data.input <- data_sub[["RNA"]]$data 

In [7]:
labels <- Idents(data_sub)
meta <- data.frame(labels = Idents(data_sub))
rownames(meta) <- colnames(data_sub)

coords <- data_sub@meta.data[, c("X", "Y")]
rownames(coords) <- rownames(data_sub@meta.data)

common_cells <- Reduce(intersect, list(colnames(data.input), rownames(meta), rownames(coords)))
data.input <- data.input[, common_cells]
meta       <- meta[common_cells, , drop = FALSE]
coords     <- coords[common_cells, , drop = FALSE]

spatial.factors <- data.frame(
  ratio = rep(1, length(common_cells)),
  tol   = rep(1, length(common_cells))
)
rownames(spatial.factors) <- common_cells

In [8]:
data_sub[["RNA"]]

data.input <- data_sub[["RNA"]]$data
dim(data.input)             # Should be 4960 × 12398
table(sub("(_\\d+)$", "", colnames(data.input)))

Assay (v5) data with 4960 features for 10292 cells
Top 10 variable features:
 REN, MYH1, LDB3, NDNF, PYGM, SLC4A1, CALB1, PODXL, ACTN2, UMOD 
Layers:
 counts, data, scale.data 

[1]  4960 10292


  BS2_61615A1__20240803  BS22_12012A1__20240803 BS24-M35359A1__20241025 
                   2387                    3430                    1977 
BS24-R31519A2__20241025 
                   2498 

In [9]:
head(data_sub@meta.data)

,orig.ident,nCount_RNA,nFeature_RNA,id,X,Y,npts,area,perimeter,sample_id,⋯,RNA_snn_res.1.2,RNA_snn_res.1.4,RNA_snn_res.1.6,niche_label,snn_0.2,snn_0.4,snn_0.6,snn_0.8,snn_1,niche_label_fine
,<fct>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,⋯,<fct>,<fct>,<fct>,<chr>,<fct>,<fct>,<fct>,<fct>,<fct>,<chr>
BS24-M35359A1__20241025_1,BS24-M35359A1,2996,1423,BS24-M35359A1__20241025_1,4876.864,916.0432,19,3996.1559,367.6549,BS24-M35359A1__20241025,⋯,4,2,0,Thick Ascending Limb,2,3,4,7,2,Thick Ascending Limb
BS24-M35359A1__20241025_2,BS24-M35359A1,1222,761,BS24-M35359A1__20241025_2,5089.547,1008.4130,5,895.5059,129.8797,BS24-M35359A1__20241025,⋯,2,1,7,Proximal Tubule,0,1,6,3,3,Proximal Tubule
BS24-M35359A1__20241025_3,BS24-M35359A1,7577,2104,BS24-M35359A1__20241025_3,5126.000,821.0908,16,2551.1956,251.6381,BS24-M35359A1__20241025,⋯,9,3,6,Collecting Duct,3,4,5,6,6,Collecting Duct
BS24-M35359A1__20241025_4,BS24-M35359A1,11210,2190,BS24-M35359A1__20241025_4,4754.545,915.3857,16,1892.0644,211.7863,BS24-M35359A1__20241025,⋯,17,17,16,Thick Ascending Limb,2,3,4,5,2,Thick Ascending Limb
BS24-M35359A1__20241025_5,BS24-M35359A1,2151,1166,BS24-M35359A1__20241025_5,4831.060,922.2174,13,969.9319,131.8925,BS24-M35359A1__20241025,⋯,10,5,1,Fibrosis & Interstitium,0,0,0,0,9,Low QC Fibrosis
BS24-M35359A1__20241025_6,BS24-M35359A1,5020,1750,BS24-M35359A1__20241025_6,5087.002,885.4119,9,630.0071,113.8104,BS24-M35359A1__20241025,⋯,9,3,6,Collecting Duct,3,4,5,6,6,Collecting Duct


In [11]:
createCellChat

function (object, meta = NULL, group.by = NULL, datatype = c("RNA", 
    "spatial"), coordinates = NULL, spatial.factors = NULL, assay = NULL, 
    do.sparse = T) 
{
    datatype <- match.arg(datatype)
    if (inherits(x = object, what = c("matrix", "Matrix", "dgCMatrix", 
        "dgRMatrix", "CsparseMatrix"))) {
        print("Create a CellChat object from a data matrix")
        data <- object
        if (is.null(group.by)) {
            group.by <- "labels"
        }
    }
    if (is(object, "Seurat")) {
        .error_if_no_Seurat()
        print("Create a CellChat object from a Seurat object")
        if (is.null(assay)) {
            assay = Seurat::DefaultAssay(object)
            if (assay == "integrated") {
                warning("The data in the `integrated` assay is not suitable for CellChat analysis! Please use the `RNA`, `SCT` or `Spatial` assay! ")
            }
            cat(paste0("The `data` slot in the default assay is used. The default assay is ", 
                assay), "\n")
        }
        if (packageVersion("Seurat") < "5.0.0") {
            data <- object[[assay]]@data
        }
        else {
            data <- object[[assay]]$data
        }
        if (min(data) < 0) {
            stop("The data matrix contains negative values. Please ensure the normalized data matrix is used.")
        }
        if (is.null(meta)) {
            cat("The `meta.data` slot in the Seurat object is used as cell meta information", 
                "\n")
            meta <- object@meta.data
            meta$ident <- Seurat::Idents(object)
        }
        if (is.null(group.by)) {
            group.by <- "ident"
        }
        if (datatype %in% c("spatial")) {
            if (is.null(coordinates)) {
                coordinates <- Seurat::GetTissueCoordinates(object, 
                  scale = NULL, cols = c("imagerow", "imagecol"))
            }
        }
    }
    if (is(object, "SingleCellExperiment")) {
        print("Create a CellChat object from a SingleCellExperiment object")
        if (is.null(assay)) {
            assay = "logcounts"
        }
        if (assay %in% SummarizedExperiment::assayNames(object)) {
            cat(paste0("The data in the ", assay, " assay is used! "), 
                "\n")
            data <- SummarizedExperiment::assay(object, assay)
        }
        else {
            stop("SingleCellExperiment object must contain an assay named `logcounts` or the input assay name! Please check the available assaynames via `assayNames(object)`. \n")
        }
        if (is.null(meta)) {
            cat("The `colData` assay in the SingleCellExperiment object is used as cell meta information", 
                "\n")
            meta <- as.data.frame(SingleCellExperiment::colData(object))
        }
        if (is.null(group.by)) {
            stop("`group.by` should be defined!")
        }
    }
    if (!inherits(x = data, what = c("dgCMatrix")) & do.sparse) {
        if (inherits(x = data, what = c("dgRMatrix"))) {
            data <- as(data, "CsparseMatrix")
        }
        data <- as(data, "dgCMatrix")
    }
    if (!is.null(meta)) {
        if (inherits(x = meta, what = c("matrix", "Matrix", "DataFrame"))) {
            meta <- as.data.frame(x = meta)
        }
        if (!is.data.frame(meta)) {
            stop("The input `meta` should be a data frame")
        }
        if (!identical(rownames(meta), colnames(data))) {
            cat("The cell barcodes in 'meta' is ", head(rownames(meta)), 
                "\n")
            warning("The cell barcodes in 'meta' is different from those in the used data matrix.\n              We now simply assign the colnames in the data matrix to the rownames of 'mata'!")
            rownames(meta) <- colnames(data)
        }
    }
    else {
        meta <- data.frame()
    }
    if (datatype %in% c("spatial")) {
        if (ncol(coordinates) == 2) {
            colnames(coordinates) <- c("x_cent", "y_cent")
        }
        else {
            sto

In [13]:
head(spatial.factors)

,ratio,tol
,<dbl>,<dbl>
BS24-M35359A1__20241025_1,1,1
BS24-M35359A1__20241025_2,1,1
BS24-M35359A1__20241025_3,1,1
BS24-M35359A1__20241025_4,1,1
BS24-M35359A1__20241025_5,1,1
BS24-M35359A1__20241025_6,1,1


In [14]:
head(coords)

,X,Y
,<dbl>,<dbl>
BS24-M35359A1__20241025_1,4876.864,916.0432
BS24-M35359A1__20241025_2,5089.547,1008.4130
BS24-M35359A1__20241025_3,5126.000,821.0908
BS24-M35359A1__20241025_4,4754.545,915.3857
BS24-M35359A1__20241025_5,4831.060,922.2174
BS24-M35359A1__20241025_6,5087.002,885.4119


In [17]:
head(data_sub@meta.data)

,orig.ident,nCount_RNA,nFeature_RNA,id,X,Y,npts,area,perimeter,sample_id,⋯,RNA_snn_res.1.2,RNA_snn_res.1.4,RNA_snn_res.1.6,niche_label,snn_0.2,snn_0.4,snn_0.6,snn_0.8,snn_1,niche_label_fine
,<fct>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,⋯,<fct>,<fct>,<fct>,<chr>,<fct>,<fct>,<fct>,<fct>,<fct>,<chr>
BS24-M35359A1__20241025_1,BS24-M35359A1,2996,1423,BS24-M35359A1__20241025_1,4876.864,916.0432,19,3996.1559,367.6549,BS24-M35359A1__20241025,⋯,4,2,0,Thick Ascending Limb,2,3,4,7,2,Thick Ascending Limb
BS24-M35359A1__20241025_2,BS24-M35359A1,1222,761,BS24-M35359A1__20241025_2,5089.547,1008.4130,5,895.5059,129.8797,BS24-M35359A1__20241025,⋯,2,1,7,Proximal Tubule,0,1,6,3,3,Proximal Tubule
BS24-M35359A1__20241025_3,BS24-M35359A1,7577,2104,BS24-M35359A1__20241025_3,5126.000,821.0908,16,2551.1956,251.6381,BS24-M35359A1__20241025,⋯,9,3,6,Collecting Duct,3,4,5,6,6,Collecting Duct
BS24-M35359A1__20241025_4,BS24-M35359A1,11210,2190,BS24-M35359A1__20241025_4,4754.545,915.3857,16,1892.0644,211.7863,BS24-M35359A1__20241025,⋯,17,17,16,Thick Ascending Limb,2,3,4,5,2,Thick Ascending Limb
BS24-M35359A1__20241025_5,BS24-M35359A1,2151,1166,BS24-M35359A1__20241025_5,4831.060,922.2174,13,969.9319,131.8925,BS24-M35359A1__20241025,⋯,10,5,1,Fibrosis & Interstitium,0,0,0,0,9,Low QC Fibrosis
BS24-M35359A1__20241025_6,BS24-M35359A1,5020,1750,BS24-M35359A1__20241025_6,5087.002,885.4119,9,630.0071,113.8104,BS24-M35359A1__20241025,⋯,9,3,6,Collecting Duct,3,4,5,6,6,Collecting Duct


In [18]:
#Create a CellChat object
cellchat <- createCellChat(
  object = data.input,      # your normalized expression matrix
  meta = data_sub@meta.data,              # includes "labels"
  group.by = "niche_label_fine",      # what to cluster/group by
  datatype = "spatial",     
  coordinates = as.matrix(coords),     # your custom X/Y
  do.sparse = T
#  spatial.factors = spatial.factors
)

[1] "Create a CellChat object from a data matrix"


ERROR: Error in createCellChat(object = data.input, meta = data_sub@meta.data, : spatial.factors with colnames `ratio` and `tol` should be provided!


In [43]:
#Set the ligand-receptor interaction database
library(CellChat)
data(CellChatDB.human)
CellChatDB.use <- subsetDB(CellChatDB.human, search = "Secreted Signaling", key = "annotation")
cellchat@DB <- CellChatDB.use

#Preprocessing the expression data for cell-cell communication analysis
cellchat <- subsetData(cellchat)
future::plan("multisession", workers = 12)
cellchat <- identifyOverExpressedGenes(cellchat, do.fast = FALSE)
cellchat <- identifyOverExpressedInteractions(cellchat)

The number of highly variable ligand-receptor pairs used for signaling inference is 397 


In [ ]:
#Part II: Inference of cell-cell communication network
#Compute the communication probability and infer cellular communication network
library(future)
options(future.globals.maxSize = 2 * 1024^3) # 2 GB
ptm = Sys.time()
cellchat <- computeCommunProb(
  cellchat, 
  raw.use = TRUE,  
  type = "truncatedMean", 
  trim = 0.1,
  distance.use = FALSE,           
  interaction.range = 250,      
  scale.distance = NULL,
  contact.dependent = TRUE,      
  contact.range = 100            
)

cellchat <- filterCommunication(cellchat, min.cells = 10)


Attaching package: ‘future’


The following objects are masked from ‘package:igraph’:

    %->%, %<-%




truncatedMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on spatial transcriptomics data without distance values as constraints of the computed communication probability <<< [2025-09-25 09:25:01.509073]"
Molecules of the input L-R pairs are diffusible. Run CellChat in a diffusion manner based on the `interaction.range`.


In [ ]:
cellchat

In [ ]:
cellchat <- computeCommunProbPathway(cellchat)
cellchat <- aggregateNet(cellchat)
execution.time = Sys.time() - ptm
print(as.numeric(execution.time, units = "secs"))

saveRDS(cellchat, file = paste0("test_nichelabels_control_processed.rds"))